[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [PyTorch and TensorFlow, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pytorch-and-tensorflow-deep-dive.html)

# Batches and Data Pipelines


## What you will be able to do

Feed a model a few examples at a time with PyTorch's `DataLoader` and TensorFlow's `tf.data`,
shuffled into a new order every epoch, and into the same sequence of orders on every run, and train
with those batches in both frameworks.

## The idea

### The problem

Every training step so far has computed its loss and gradient from all 800 training stations at
once. That works because 800 rows of four numbers fit in memory many times over. A collection of a
million photographs does not, and computing one gradient from all of them for every step would take
a long time even where it fit.

A gradient computed from 64 stations chosen at random is an estimate of the one computed from all
800, and it costs about a twelfth as much. So training takes a step after every few dozen
examples, and a pass through the data makes 13 steps instead of one. Choosing those examples by
hand, in a new order each pass, takes code that is easy to get subtly wrong, and both frameworks
provide it.

### A formal definition

> A **batch** is the group of examples one training step computes its loss and gradient from, and
> an **epoch** is one pass through the training data, batch by batch. A **data pipeline** produces
> the batches. It holds the examples, shuffles their order for each epoch, groups them, and can
> change each example on the way. In PyTorch it is a **`Dataset`** read by a **`DataLoader`**, and
> in TensorFlow a **`tf.data.Dataset`**.

### Why it works that way

A PyTorch `Dataset` tells the loader how many examples it holds and what example number `i` is. The `DataLoader` decides which numbers go into each batch, asks the dataset for each one, and
stacks the answers into one tensor per item, which needs every example to have the same shape. With
`shuffle=True` it draws a new order at the start of every epoch, from a random generator you can
seed.

A `tf.data.Dataset` is a chain of steps, such as making one from arrays, shuffling it, batching it
and mapping a function over it, and each step returns a new dataset. Nothing is computed until something reads from
the end of the chain. Its `shuffle` keeps a buffer of examples and draws from that buffer at
random, refilling it as it goes, so it mixes examples only within the distance the buffer covers,
and a buffer as large as the data mixes all of it.

Shuffling matters because a batch should be a fair sample. Data stored in an order, by date or by
place, gives batches that each see one corner of it, and a model that keeps chasing whichever corner
it saw last.

### Where this shows up

The **Training Loops** notebook trains with a `DataLoader` and checks a validation set after each
epoch. The **Convolutional Networks** and **Recurrent Networks** notebooks train on images and
sequences in shuffled batches, drawn there with `torch.randperm` and Keras's `batch_size`.

### What this notebook covers

- a `TensorDataset` and a `Dataset` class of your own
- batches from a `DataLoader`, a new order every epoch, and `drop_last`
- the effect of the batch size on how far 20 epochs get
- the same pipeline in `tf.data`, with `shuffle`, `batch` and `map`, and Keras's `fit` reading it
- four ways a pipeline goes wrong

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read it,
and read the output underneath it. Everything from Setup onward is where you start running things.

```python
import torch
from torch.utils.data import DataLoader, TensorDataset

numbers = TensorDataset(torch.arange(10))
loader = DataLoader(numbers, batch_size=4, shuffle=True, generator=torch.Generator().manual_seed(0))
for (batch,) in loader:
    print(batch)
```

```
tensor([3, 7, 5, 2])
tensor([0, 8, 1, 6])
tensor([9, 4])
```

The ten examples come in batches of four, in an order the seeded generator chose. Every example
appears once, and the last batch holds the two left over.

## Setup

Three libraries, PyTorch's data tools, a way to quiet TensorFlow while it loads, and the stations.

- `numpy` builds the stations
- `torch` is PyTorch
- `DataLoader`, `Dataset` and `TensorDataset` are PyTorch's data pipeline
- `pad_sequence` pads examples of different lengths to one length, used once, in Common errors
- `tensorflow` is TensorFlow, whose pipeline is `tf.data`, and `keras` its library of layers and
  training tools
- `contextlib` and `os` hide the messages TensorFlow prints while it loads
- `keras.utils.set_random_seed(0)` sets TensorFlow's global seed, which every `tf.data` shuffle
  combines with its own `seed`

TensorFlow's C++ code prints several lines as it loads, each stamped with the time and a process
number, so no two runs would show the same output. Setting `TF_CPP_MIN_LOG_LEVEL` to 3 hides what
TensorFlow logs once it has started, and `c_messages_hidden` sends the lines it prints while it
starts to `os.devnull`. The cell prints PyTorch's version without the part after `+`, which names
the build.

`station_network_arrays` makes the 1,200 made-up weather stations of the **scikit-learn, Deep Dive**
guide, drawn from a fixed seed, so they are the same on every run. `features` holds each station's
latitude, longitude, elevation and distance from the coast as 32-bit floats, `jan_mean` its January
mean in degrees Celsius, and `snow_on_may_1` a 1 for a station with snow on 1 May and a 0 otherwise.
The cell splits the stations into 800 for training, 200 for validation and 200 for testing, and
standardizes the features with the training stations' means and standard deviations into `scaled`.

In [1]:
import contextlib
import os

import numpy as np
import torch
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset, TensorDataset


@contextlib.contextmanager
def c_messages_hidden():
    """Send what C and C++ code writes to the error stream to os.devnull while the block runs."""
    saved = os.dup(2)
    devnull = os.open(os.devnull, os.O_WRONLY)
    os.dup2(devnull, 2)
    try:
        yield
    finally:
        os.dup2(saved, 2)
        os.close(devnull)
        os.close(saved)


os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
with c_messages_hidden():
    import tensorflow as tf
    import keras


def station_network_arrays(n=1200, seed=2025):
    """The made-up stations of the scikit-learn guide, as arrays: four features and two targets."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n)
    longitude = 5 + 1.5 * (latitude - 58) + rng.uniform(0, 5, n)
    coast_km = np.minimum(rng.exponential(25, n), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n) + 3 * coast_km, 0, 1800)
    shift = np.where(coast_km < 3, 0.8,
            np.where(elevation > 700, -0.5,
            np.where((coast_km < 30) & (rng.uniform(size=n) < 0.6), 0.3, 0.0)))
    rng.choice(4, n, p=[0.4, 0.3, 0.1, 0.2])   # each station's owner, which this guide leaves out
    rng.integers(1950, 2025, n)                 # and the year it was installed
    jan_mean = (1.5 - 0.55 * (latitude - 58) - 0.006 * elevation - 4.0 * (1 - np.exp(-coast_km / 40))
                + shift + rng.normal(0, 0.9, n))
    may_mean = jan_mean + 8.5 + 0.004 * coast_km + rng.normal(0, 1.2, n)
    features = np.column_stack([latitude.round(2), longitude.round(2), elevation.round(0), coast_km.round(1)])
    return features.astype("float32"), jan_mean.round(1).astype("float32"), (may_mean < 2.5).astype("float32")


FEATURES = ["latitude", "longitude", "elevation_m", "coast_km"]
features, jan_mean, snow_on_may_1 = station_network_arrays()
order = np.random.default_rng(0).permutation(len(features))
train_rows, valid_rows, test_rows = order[:800], order[800:1000], order[1000:]
mean, std = features[train_rows].mean(axis=0), features[train_rows].std(axis=0)
scaled = (features - mean) / std

print("torch", torch.__version__.split("+")[0], "| tensorflow", tf.__version__, "| keras", keras.__version__,
      "|", len(features), "stations")
keras.utils.set_random_seed(0)

torch 2.11.0 | tensorflow 2.21.0 | keras 3.13.2 | 1200 stations


## Worked examples

### A dataset of the training stations

`TensorDataset` takes tensors with the same first dimension and makes example `i` the `i`-th row of
each.

In [2]:
X_train = torch.from_numpy(scaled[train_rows])
y_train = torch.from_numpy(jan_mean[train_rows]).unsqueeze(1)
X_test = torch.from_numpy(scaled[test_rows])
y_test = torch.from_numpy(jan_mean[test_rows]).unsqueeze(1)

train_set = TensorDataset(X_train, y_train)
print(len(train_set), "examples")
print(train_set[0])

800 examples
(tensor([ 1.2453,  0.8326, -0.5109,  0.6045]), tensor([-8.1000]))


Each example is a pair, a station's four standardized features and its January mean.

### Batches from a DataLoader

In [3]:
loader = DataLoader(train_set, batch_size=64, shuffle=True, generator=torch.Generator().manual_seed(0))
print(len(loader), "batches:", [len(xb) for xb, yb in loader])

features_batch, targets_batch = next(iter(loader))
print(features_batch.shape, targets_batch.shape)

13 batches: [64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 32]
torch.Size([64, 4]) torch.Size([64, 1])


800 stations make twelve full batches of 64 and a last batch of 32. Each batch is the
examples' features stacked into one (64, 4) tensor and their targets into one (64, 1) tensor.
`drop_last=True` leaves the short batch out, which keeps every step's batch the same size.

In [4]:
print([len(xb) for xb, yb in DataLoader(train_set, batch_size=64, drop_last=True)])

[64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64]


### A batch's gradient against the whole set's

The cosine similarity of two gradients is 1 when they point the same way and 0 when they are at
right angles. Here is the gradient of a new network's loss computed from all 800 training stations,
against the gradients from the first three batches of 64.

In [5]:
torch.manual_seed(0)
net = torch.nn.Sequential(torch.nn.Linear(4, 32), torch.nn.ReLU(), torch.nn.Linear(32, 1))


def gradient(xb, yb):
    """The network's gradient on one batch, as one long vector."""
    net.zero_grad()
    torch.nn.functional.mse_loss(net(xb), yb).backward()
    return torch.cat([p.grad.ravel() for p in net.parameters()])


full = gradient(X_train, y_train)
for i, (xb, yb) in enumerate(loader):
    if i == 3:
        break
    print("batch", i, "cosine similarity", round(torch.nn.functional.cosine_similarity(full, gradient(xb, yb), dim=0).item(), 3))

batch 0 cosine similarity 0.988
batch 1 cosine similarity 0.988
batch 2 cosine similarity 0.996


The three batches give cosine similarities of 0.988, 0.988 and 0.996, within nine degrees of the
full gradient. The network is new, and its predictions are all off from the January means in the
same direction, so every batch agrees on the direction that lowers the loss. After some training the
full gradient is small, and each batch's gradient is mostly the noise of its own 64 stations, as the
next cell shows.

In [6]:
first_size = full.norm().item()
optimizer = torch.optim.Adam(net.parameters(), lr=0.01)
for step in range(200):
    optimizer.zero_grad()
    torch.nn.functional.mse_loss(net(X_train), y_train).backward()
    optimizer.step()

full = gradient(X_train, y_train)
print(f"size of the full gradient: {first_size:.2f} for the new network, {full.norm().item():.2f} after 200 steps")
for i, (xb, yb) in enumerate(loader):
    if i == 3:
        break
    print("after 200 steps, batch", i, "cosine similarity",
          round(torch.nn.functional.cosine_similarity(full, gradient(xb, yb), dim=0).item(), 3))

size of the full gradient: 21.67 for the new network, 0.15 after 200 steps
after 200 steps, batch 0 cosine similarity 0.336
after 200 steps, batch 1 cosine similarity 0.312
after 200 steps, batch 2 cosine similarity 0.724


After 200 steps the full gradient has shrunk from 21.67 to 0.15, and the three batches give cosine
similarities of 0.336, 0.312 and 0.724, two of them more than 70 degrees away from the full gradient.
A step on 64 stations now goes partly the way the whole set would and partly the way those 64
stations alone point.

### A new order every epoch

A dataset of the row numbers 0 to 799 shows which examples each batch holds. Reading the loader again
starts a new epoch.

In [7]:
row_numbers = TensorDataset(torch.arange(800))
shuffled = DataLoader(row_numbers, batch_size=8, shuffle=True, generator=torch.Generator().manual_seed(0))

for epoch in range(2):
    (first_batch,) = next(iter(shuffled))
    print("epoch", epoch, "starts with", first_batch.tolist())
(unshuffled_batch,) = next(iter(DataLoader(row_numbers, batch_size=8)))
print("without shuffle:", unshuffled_batch.tolist())

epoch 0 starts with [533, 51, 507, 176, 307, 783, 699, 271]
epoch 1 starts with [472, 723, 763, 30, 432, 564, 679, 774]
without shuffle: [0, 1, 2, 3, 4, 5, 6, 7]


Each epoch starts with a different batch, and without `shuffle` the rows come in order. Seeding the
generator makes the sequence of orders the same on every run of the notebook, so the first epoch
starts with rows 533, 51 and 507 every time.

### A Dataset class of your own

A dataset can be any class with `__len__` and `__getitem__`. This one holds row numbers and
standardizes a station's features only when the loader asks for it, which is how a dataset of
images reads each file only when it is needed.

In [8]:
class StationDataset(Dataset):
    """Stations by row number: standardized features and the January mean, made when asked for."""

    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        row = self.rows[i]
        return torch.from_numpy((features[row] - mean) / std), torch.tensor([jan_mean[row]])


stations_by_row = StationDataset(train_rows)
print(len(stations_by_row), stations_by_row[0])
features_batch, targets_batch = next(iter(DataLoader(stations_by_row, batch_size=4)))
print(features_batch.shape, targets_batch.shape, features_batch.dtype)

800 (tensor([ 1.2453,  0.8326, -0.5109,  0.6045]), tensor([-8.1000]))
torch.Size([4, 4]) torch.Size([4, 1]) torch.float32


Example 0 is the same pair `TensorDataset` gave, and the loader stacks four of them into the
same shapes.

### How far 20 epochs get at each batch size

`train_with_batches` trains the 4-32-1 network of the **Why PyTorch and TensorFlow** notebook with
Adam, from the same seed, for 20 epochs, and returns the number of steps and the final losses on
the training and test stations.

In [9]:
def train_with_batches(batch_size, epochs=20):
    """Train a fresh 4-32-1 network on batches of the training stations; return steps and losses."""
    torch.manual_seed(0)
    net = torch.nn.Sequential(torch.nn.Linear(4, 32), torch.nn.ReLU(), torch.nn.Linear(32, 1))
    optimizer = torch.optim.Adam(net.parameters(), lr=0.01)
    loader = DataLoader(train_set, batch_size=batch_size, shuffle=True,
                        generator=torch.Generator().manual_seed(0))
    steps = 0
    for epoch in range(epochs):
        for xb, yb in loader:
            optimizer.zero_grad()
            torch.nn.functional.mse_loss(net(xb), yb).backward()
            optimizer.step()
            steps += 1
    with torch.no_grad():
        train_loss = torch.nn.functional.mse_loss(net(X_train), y_train).item()
        test_loss = torch.nn.functional.mse_loss(net(X_test), y_test).item()
    return steps, round(train_loss, 3), round(test_loss, 3)


for batch_size in (800, 64, 16):
    steps, train_loss, test_loss = train_with_batches(batch_size)
    print(f"batch size {batch_size:3}: {steps:4} steps, training loss {train_loss}, test loss {test_loss}")

batch size 800:   20 steps, training loss 12.261, test loss 13.674
batch size  64:  260 steps, training loss 0.886, test loss 0.985
batch size  16: 1000 steps, training loss 0.8, test loss 0.968


With every station in one batch, 20 epochs are 20 steps, and the loss is still above 12. Batches of
64 make 13 steps an epoch, 260 in all, and reach a test loss of 0.985, close to the 0.96 that the
same network reached in the **Why PyTorch and TensorFlow** notebook after 500 full steps. Batches of
16 make 1,000 steps and end at 0.968. A step on a smaller batch is cheaper and noisier, and the
right size is a trade-off between the two that depends on the data and the machine.

### The same pipeline in tf.data

`from_tensor_slices` takes a pair of arrays and makes example `i` the `i`-th row of each, as
`TensorDataset` does, and `element_spec` describes one example.

In [10]:
train_tf = tf.data.Dataset.from_tensor_slices((scaled[train_rows], jan_mean[train_rows][:, None]))
print(train_tf.element_spec)
print(int(train_tf.cardinality()), "examples")

batches = train_tf.shuffle(800, seed=0).batch(64)
print(int(batches.cardinality()), "batches:", [int(x.shape[0]) for x, y in batches])

(TensorSpec(shape=(4,), dtype=tf.float32, name=None), TensorSpec(shape=(1,), dtype=tf.float32, name=None))
800 examples
13 batches: [64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 32]


`shuffle(800, seed=0)` keeps a buffer as large as the training set, so it mixes all of it.

In [11]:
row_numbers_tf = tf.data.Dataset.range(800).shuffle(800, seed=0).batch(8)
for epoch in range(2):
    print("epoch", epoch, "starts with", next(iter(row_numbers_tf)).numpy().tolist())

fixed = tf.data.Dataset.range(800).shuffle(800, seed=0, reshuffle_each_iteration=False).batch(8)
print("the same order each time:", next(iter(fixed)).numpy().tolist() == next(iter(fixed)).numpy().tolist())

epoch 0 starts with [634, 435, 543, 727, 793, 442, 713, 745]
epoch 1 starts with [590, 525, 306, 28, 272, 771, 629, 650]
the same order each time: True


As with the `DataLoader`, each read starts a new order. The order comes from the shuffle's own
`seed` together with TensorFlow's global seed, which Setup sets, so the sequence of orders is the
same on every run. `reshuffle_each_iteration=False` keeps one order for every epoch, so every epoch
would meet the batches in the same sequence.

### Changing each example with map

`map` applies a function to every example as it passes. This pipeline starts from the raw features
and standardizes them on the way, as `StationDataset` did, and `prefetch` lets the pipeline prepare
the next batch while the model works on the current one.

In [12]:
raw_tf = tf.data.Dataset.from_tensor_slices((features[train_rows], jan_mean[train_rows][:, None]))
standardized = raw_tf.map(lambda x, y: ((x - mean) / std, y))
pipeline = standardized.shuffle(800, seed=0).batch(64).prefetch(tf.data.AUTOTUNE)

first_raw, _ = next(iter(raw_tf))
first_standardized, _ = next(iter(standardized))
print(first_raw.numpy(), "->", first_standardized.numpy().round(4))
print(pipeline.element_spec)

[ 69.05  21.98 195.    38.5 ] -> [ 1.2453  0.8326 -0.5109  0.6045]
(TensorSpec(shape=(None, 4), dtype=tf.float32, name=None), TensorSpec(shape=(None, 1), dtype=tf.float32, name=None))


The first raw station becomes the same four standardized numbers that `scaled` holds for it.
A batched dataset's `element_spec` gives the batch dimension as `None`, since without
`drop_remainder=True` the last batch can be shorter.

### Keras's fit reading a dataset

Given a `tf.data` pipeline, `fit` takes its batches from it, so it gets no `batch_size`. An epoch is
one pass through the pipeline.

In [13]:
keras.utils.set_random_seed(0)
net_keras = keras.Sequential([keras.Input((4,)), keras.layers.Dense(32, activation="relu"), keras.layers.Dense(1)])
net_keras.compile(optimizer=keras.optimizers.Adam(0.01), loss="mse")
history = net_keras.fit(pipeline, epochs=20, verbose=0)

print("training loss", round(net_keras.evaluate(scaled[train_rows], jan_mean[train_rows], verbose=0), 3),
      "| test loss", round(net_keras.evaluate(scaled[test_rows], jan_mean[test_rows], verbose=0), 3))

training loss 0.843 | test loss 0.961


The Keras network reaches a training loss of 0.843 and a test loss of 0.961 after 20 epochs of
batches of 64, against 0.886 and 0.985 for PyTorch's network with the same batch size. Both losses
are measured after training, on all the training stations and on the test stations, as
`train_with_batches` measures PyTorch's. The two networks start from different random weights and
shuffle in different orders.

## Your turn

Five tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pytorch-and-tensorflow-deep-dive/09-batches-and-data-pipelines-solutions.ipynb).

**1.** Make a `DataLoader` of `train_set` with `batch_size=100`, and print how many batches it makes and the size of each.

In [14]:
# your code here

**2.** Print the first batch of eight row numbers for three epochs from a shuffling `DataLoader` of `row_numbers` whose generator is seeded with 1.

In [15]:
# your code here

**3.** Write a `Dataset` class whose examples are a station's standardized features and its snow label as a one-element float tensor, and print the shapes of one batch of 16.

In [16]:
# your code here

**4.** Use `train_with_batches` to train with batches of 32 and of 128, and compare their steps and losses with the three printed above.

In [17]:
# your code here

**5.** Build a `tf.data` pipeline from the raw features of the training stations that standardizes them with `map`, shuffles with `seed=1` and batches by 32, train the Keras network on it for 20 epochs from `keras.utils.set_random_seed(0)`, and print the test loss.

In [18]:
# your code here

## Common errors

### `AssertionError: Size mismatch between tensors`

In [19]:
TensorDataset(X_train, y_test)

AssertionError: Size mismatch between tensors

`X_train` has 800 rows and `y_test` 200, and `TensorDataset` needs the same number in every tensor
it is given, since example `i` takes row `i` of each. The message names no sizes, so print the
shapes. Here the targets came from the wrong split, and `y_train` is the one that matches.

### `ValueError: Dimensions 800 and 200 are not compatible`

In [20]:
tf.data.Dataset.from_tensor_slices((scaled[train_rows], jan_mean[test_rows]))

ValueError: Dimensions 800 and 200 are not compatible

This is TensorFlow's version of the same mistake, and its message names the two sizes.

### `RuntimeError: stack expects each tensor to be equal size, but got [7] at entry 0 and [6] at entry 1`

The cell makes up a week of daily temperatures for three stations from each one's January mean, and
the second station misses its last day.

In [21]:
pattern = np.array([0.4, -1.2, 0.3, 1.1, -0.6, 0.2, -0.2], dtype="float32")
weeks = [torch.from_numpy(jan_mean[row] + pattern) for row in train_rows[:3]]
weeks[1] = weeks[1][:6]
print([len(week) for week in weeks])
next(iter(DataLoader(weeks, batch_size=3)))

[7, 6, 7]


RuntimeError: stack expects each tensor to be equal size, but got [7] at entry 0 and [6] at entry 1

The `DataLoader` stacks the examples of a batch into one tensor, and a tensor is a rectangle, so a
batch of a seven-day week and a six-day one cannot be stacked. `entry 0` and `entry 1` are the
positions in the batch. A `collate_fn` replaces the stacking, and this one pads every example to the
longest with 0 and returns the true lengths beside them, so that a model can tell padding from
readings.

In [22]:
def pad_batch(examples):
    """Pad examples of different lengths to the longest, and return their lengths too."""
    return pad_sequence(examples, batch_first=True), torch.tensor([len(e) for e in examples])


padded, lengths = next(iter(DataLoader(weeks, batch_size=3, collate_fn=pad_batch)))
print(padded.numpy().round(1))
print(lengths)

[[-7.7 -9.3 -7.8 -7.  -8.7 -7.9 -8.3]
 [-5.2 -6.8 -5.3 -4.5 -6.2 -5.4  0. ]
 [-5.5 -7.1 -5.6 -4.8 -6.5 -5.7 -6.1]]
tensor([7, 6, 7])


### No error, and an order that barely changes

The cell sorts the training stations from north to south and shuffles them with a buffer of 10 and
with a buffer as large as the data.

In [23]:
keras.utils.set_random_seed(0)
north_to_south = np.sort(features[train_rows, 0])[::-1].copy()
for buffer in (10, 800):
    batches = list(tf.data.Dataset.from_tensor_slices(north_to_south).shuffle(buffer, seed=0).batch(64))
    first, last = batches[0].numpy(), batches[-1].numpy()
    print(f"buffer {buffer:3}: first batch spans latitudes {first.min():.2f} to {first.max():.2f}, "
          f"last batch {last.min():.2f} to {last.max():.2f}")
print(f"all training stations: {north_to_south.min():.2f} to {north_to_south.max():.2f}")

buffer  10: first batch spans latitudes 69.80 to 70.98, last batch 58.01 to 58.59
buffer 800: first batch spans latitudes 58.09 to 70.66, last batch 58.66 to 70.89
all training stations: 58.01 to 70.98


A buffer of 10 holds only the next ten stations, so the first batch of 64 is drawn from the first
73 in the sorted order, all of them in the far north, and the last batch from the far south, 58.01
to 58.59. The pipeline raises nothing, and a model trained on it would meet the north at the start
of the epoch and the south at the end. Give `shuffle` a buffer as large as the data when the data
fits in memory, as these stations do.

## Recap

- You can make a `TensorDataset`, or a `Dataset` class with `__len__` and `__getitem__`, and read it in
  shuffled batches with a seeded `DataLoader`.
- You can build the same pipeline with `tf.data`, from `from_tensor_slices` through `shuffle`, `map`
  and `batch`, and hand it to Keras's `fit`.
- You can choose a batch size, and see how it changes the number of steps an epoch makes.
- You can pad examples of different lengths with a `collate_fn`, and shuffle with a buffer that mixes
  all of the data.

## What is next

The **Training Loops** notebook puts the pieces together: batches, a loss, an optimizer and a check
on the validation stations after every epoch, written out in PyTorch and run by Keras's `fit`.

---

&#8592; **Previous:** [Losses and Optimizers](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pytorch-and-tensorflow-deep-dive/08-losses-and-optimizers.ipynb)  &nbsp;·&nbsp;  [PyTorch and TensorFlow, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pytorch-and-tensorflow-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Training Loops](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pytorch-and-tensorflow-deep-dive/10-training-loops.ipynb) &#8594;
